# Statystyka dla Data Analyst/Engineer — Moduł 11: Korelacja i kowariancja

Do tej pory pytaliśmy głównie "czy średnie/proporcje się różnią?". Teraz zadamy inne
pytanie: **czy dwie zmienne ciągłe poruszają się razem?** Gdy wydatki klienta rosną, czy
jego satysfakcja też rośnie? Korelacja i kowariancja to narzędzia do mierzenia siły i
kierunku takiego związku — i fundament pod regresję liniową w Module 12.

## Spis treści
1. [Kowariancja](#sec1)
2. [Korelacja Pearsona](#sec2)
3. [Istotność statystyczna korelacji](#sec3)
4. [Korelacja Spearmana](#sec4)
5. [Korelacja to nie przyczynowość](#sec5)
6. [Macierz korelacji i mapa ciepła](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Kowariancja

**Kowariancja** mierzy, czy dwie zmienne poruszają się w tym samym kierunku (dodatnia
kowariancja: gdy jedna rośnie, druga zwykle też rośnie), w przeciwnych kierunkach
(ujemna), czy niezależnie (bliska zeru):

```
cov(X, Y) = srednia z [(x - srednia_x) * (y - srednia_y)]
```

In [ ]:
kowariancja = klienci["wydatki_miesieczne"].cov(klienci["satysfakcja"])
print(f"Kowariancja (wydatki, satysfakcja): {kowariancja:.2f}")


> ⚠️ **Kowariancja jest w praktyce bezużyteczna sama w sobie**
>
> Wartość kowariancji zależy od JEDNOSTEK obu zmiennych — gdybyśmy wyrazili wydatki w groszach zamiast złotych, kowariancja urosłaby stukrotnie, mimo że "siła" związku wcale się nie zmieniła. Kowariancja mówi Ci tylko o KIERUNKU związku (dodatnia/ujemna), nigdy o jego SILE w sposób porównywalny między różnymi parami zmiennych. Dlatego w praktyce niemal zawsze używa się korelacji (sekcja 2) — znormalizowanej wersji kowariancji.

<a id="sec2"></a>
## 2. Korelacja Pearsona

**Korelacja Pearsona** (`r`) to kowariancja podzielona przez iloczyn odchyleń
standardowych obu zmiennych — dzięki temu zawsze mieści się w przedziale `[-1, 1]`,
niezależnie od jednostek:

```
r = cov(X, Y) / (odchylenie_std_x * odchylenie_std_y)
```

- `r = 1` — idealna dodatnia zależność LINIOWA,
- `r = -1` — idealna ujemna zależność liniowa,
- `r = 0` — brak zależności LINIOWEJ (uwaga: nie znaczy "brak jakiejkolwiek zależności"!),
- wartości pośrednie — częściowa zależność liniowa, im bliżej `1`/`-1`, tym silniejsza.

In [ ]:
from scipy import stats

r, wartosc_p = stats.pearsonr(klienci["wydatki_miesieczne"], klienci["satysfakcja"])
print(f"r Pearsona: {r:.3f}")
print(f"wartosc p: {wartosc_p:.2e}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
plt.scatter(klienci["wydatki_miesieczne"], klienci["satysfakcja"], alpha=0.5)
plt.xlabel("Wydatki miesieczne")
plt.ylabel("Satysfakcja")
plt.title(f"r = {r:.3f}")
plt.show()


> 📊 **Anscombe znowu puka do drzwi**
>
> Pamiętasz kwartet Anscombe'a z Modułu 2? To ten sam problem: `r` mierzy TYLKO zależność LINIOWĄ. Silna zależność krzywoliniowa (np. paraboliczna) może dać `r` bliskie zeru, mimo wyraźnego, przewidywalnego wzorca. Zawsze narysuj wykres rozrzutu — dokładnie jak zrobiliśmy powyżej — zanim zaufasz samej liczbie `r`.

<a id="sec3"></a>
## 3. Istotność statystyczna korelacji

`stats.pearsonr` zwraca nie tylko `r`, ale też wartość p — test hipotezy, czy `r` różni
się istotnie od zera w populacji:

- H0: `rho = 0` (brak korelacji liniowej w populacji)
- H1: `rho != 0`

In [ ]:
alpha = 0.05
print(f"r={r:.3f}, p={wartosc_p:.2e}")
print("Korelacja istotna statystycznie" if wartosc_p <= alpha else "Korelacja NIEistotna statystycznie")


> ⚠️ **Istotność statystyczna to nie to samo co siła związku**
>
> Dokładnie jak w Module 7: przy dużej próbie nawet BARDZO SŁABA korelacja (np. `r=0.05`) może wyjść "istotna statystycznie". Zawsze patrz na WARTOŚĆ `r` (siła związku), nie tylko na wartość p (czy różni się od zera). Nasza korelacja wydatki-satysfakcja jest i silna, i istotna — ale to nie zawsze idzie w parze.

<a id="sec4"></a>
## 4. Korelacja Spearmana

**Korelacja Spearmana** liczy korelację Pearsona nie na surowych wartościach, tylko na
ich RANGACH (pozycji w posortowanej kolejności). Dzięki temu wykrywa dowolną zależność
**monotoniczną** (rosnącą lub malejącą), niekoniecznie liniową — i jest znacznie
odporniejsza na wartości odstające.

In [ ]:
import numpy as np

# przyklad: silnie NIEliniowa, ale monotoniczna zaleznosc
rng = np.random.default_rng(1)
x_przyklad = np.linspace(1, 10, 100)
y_przyklad = x_przyklad ** 3 + rng.normal(0, 20, size=100)  # zaleznosc szescienna

r_pearson_przyklad, _ = stats.pearsonr(x_przyklad, y_przyklad)
r_spearman_przyklad, _ = stats.spearmanr(x_przyklad, y_przyklad)

print(f"Pearson r: {r_pearson_przyklad:.3f}")
print(f"Spearman r: {r_spearman_przyklad:.3f}")

plt.figure(figsize=(7, 5))
plt.scatter(x_przyklad, y_przyklad, alpha=0.5)
plt.title(f"Pearson={r_pearson_przyklad:.3f}, Spearman={r_spearman_przyklad:.3f}")
plt.show()


Spearman powinien wyjść wyraźnie wyższy niż Pearson — bo zależność SZEŚCIENNA (`x^3`)
jest silnie monotoniczna (im większe `x`, tym większe `y`, bez wyjątków), ale nie jest
liniowa, więc Pearson jej nie doceni w pełni.

In [ ]:
r_spearman_dane, p_spearman_dane = stats.spearmanr(klienci["wydatki_miesieczne"], klienci["satysfakcja"])
print(f"Spearman (wydatki, satysfakcja): r={r_spearman_dane:.3f}, p={p_spearman_dane:.2e}")
print(f"(dla porownania, Pearson: r={r:.3f})")


<a id="sec5"></a>
## 5. Korelacja to nie przyczynowość

To być może najważniejsze zdanie całego modułu: **korelacja nie oznacza
przyczynowości**. Nawet silna, istotna statystycznie korelacja między X i Y nie mówi, że
X POWODUJE Y. Możliwe wyjaśnienia obserwowanej korelacji:

1. **X powoduje Y** (to, co często zakładamy, ale rzadko możemy udowodnić samą
   korelacją),
2. **Y powoduje X** (kierunek odwrotny — korelacja sama w sobie nie mówi, w którą
   stronę),
3. **Zmienna ukryta Z powoduje oba** (*confounding variable*) — np. wydatki mogą "wywoływać"
   satysfakcję ALBO oba mogą wynikać z jeszcze innej cechy klienta (np. zamożności),
4. **Czysty przypadek** — przy wystarczająco wielu porównywanych zmiennych, jakieś
   przypadkowe korelacje pojawią się "za darmo" (zobacz słynne, żartobliwe przykłady
   "spurious correlations" — np. konsumpcja sera na osobę koreluje z liczbą osób, które
   udusiły się pościelą).

> 🔥 **Jedyny pewny sposób na przyczynowość: eksperyment**
>
> Żeby przejść od korelacji do przyczynowości, potrzeba zwykle **eksperymentu z losowym przydziałem** (dokładnie jak nasz test A/B z Modułu 7/9!) — losowo przydzielając klientów do grup, eliminujesz wpływ zmiennych ukrytych, bo obie grupy powinny mieć podobny rozkład wszystkich innych cech. To dlatego randomizowane testy A/B są tak cenione w praktyce: dają przyczynowość, a nie tylko korelację.

<a id="sec6"></a>
## 6. Macierz korelacji i mapa ciepła

Gdy zmiennych numerycznych jest więcej niż dwie, wygodnie jest policzyć WSZYSTKIE
korelacje parami naraz — `DataFrame.corr()` robi to jedną linijką, a mapa ciepła
(*heatmap*) pozwala szybko wychwycić wzorce.

In [ ]:
import seaborn as sns

kolumny_liczbowe = ["wiek", "wydatki_miesieczne", "satysfakcja", "wydatki_przed", "wydatki_po"]
macierz_korelacji = klienci[kolumny_liczbowe].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(macierz_korelacji, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Macierz korelacji")
plt.show()


Zwróć uwagę na bardzo wysoką korelację między `wydatki_przed` i `wydatki_po` — to nie
przypadek, tylko bezpośrednia konsekwencja tego, jak zbudowaliśmy te dane w Module 8
(`wydatki_po = wydatki_przed + losowy_efekt`): kto wydawał dużo przed, zwykle wydaje dużo
i po.

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- kowariancję i dlaczego sama w sobie jest mało użyteczna (zależy od jednostek),
- korelację Pearsona (`r`) jako znormalizowaną, interpretowalną wersję kowariancji,
- istotność statystyczną korelacji i pułapkę mylenia jej z siłą związku,
- korelację Spearmana dla zależności monotonicznych, nieliniowych,
- dlaczego korelacja NIE oznacza przyczynowości (i cztery możliwe wyjaśnienia),
- macierz korelacji i mapę ciepła dla wielu zmiennych naraz.

> 📝 **Ćwiczenie 1: Korelacja wieku i wydatków**
>
> Policz korelację Pearsona między `wiek` a `wydatki_miesieczne`, razem z wartością p. Czy jest silna, słaba, istotna statystycznie?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
r_wiek, p_wiek = stats.pearsonr(klienci["wiek"], klienci["wydatki_miesieczne"])
print(f"r={r_wiek:.3f}, p={p_wiek:.4f}")
```

Wiek jest losowany calkowicie niezaleznie od wydatkow w naszych danych, wiec spodziewaj sie r bliskiego 0 i braku istotnosci.
</details>

> 📝 **Ćwiczenie 2: Spearman dla satysfakcji i wydatków przed**
>
> Policz korelację Spearmana między `wydatki_przed` a `satysfakcja`. Porównaj z korelacją Pearsona dla tej samej pary -- czy się znacząco różnią?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
r_pearson_ex2, _ = stats.pearsonr(klienci["wydatki_przed"], klienci["satysfakcja"])
r_spearman_ex2, _ = stats.spearmanr(klienci["wydatki_przed"], klienci["satysfakcja"])
print(f"Pearson: {r_pearson_ex2:.3f}, Spearman: {r_spearman_ex2:.3f}")
```
</details>

> 📝 **Ćwiczenie 3: Korelacja z outlierem**
>
> Weź kolumny `wydatki_miesieczne` i `satysfakcja`, ale dodaj SZTUCZNY punkt odstający (np. klienta z wydatkami=5000 i satysfakcja=1) do skopiowanego DataFrame. Policz r Pearsona i r Spearmana PRZED i PO dodaniu tego punktu -- która z tych dwóch miar bardziej się zmienia?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
kopia = klienci.copy()
nowy_wiersz = pd.DataFrame({"wydatki_miesieczne": [5000], "satysfakcja": [1]})
z_outlierem = pd.concat([kopia[["wydatki_miesieczne", "satysfakcja"]], nowy_wiersz], ignore_index=True)

r_przed, _ = stats.pearsonr(kopia["wydatki_miesieczne"], kopia["satysfakcja"])
r_po, _ = stats.pearsonr(z_outlierem["wydatki_miesieczne"], z_outlierem["satysfakcja"])
rs_przed, _ = stats.spearmanr(kopia["wydatki_miesieczne"], kopia["satysfakcja"])
rs_po, _ = stats.spearmanr(z_outlierem["wydatki_miesieczne"], z_outlierem["satysfakcja"])

print(f"Pearson: {r_przed:.3f} -> {r_po:.3f}")
print(f"Spearman: {rs_przed:.3f} -> {rs_po:.3f}")
```

Pearson zazwyczaj zmienia sie mocniej -- opiera sie na surowych wartosciach, wiec jeden ekstremalny punkt moze go znaczaco 'pociagnac'. Spearman, oparty na rangach, jest znacznie odporniejszy na pojedyncze odstajace obserwacje.
</details>

> 📝 **Ćwiczenie 4: Pełna macierz korelacji z wartościami p**
>
> Napisz funkcję `macierz_p(df, kolumny)`, która zwraca DataFrame z wartościami p korelacji Pearsona dla każdej pary kolumn (podpowiedź: podwójna pętla po `kolumny`, `stats.pearsonr` dla każdej pary). Uruchom ją dla tych samych kolumn co w sekcji 6.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
def macierz_p(df, kolumny):
    wynik = pd.DataFrame(index=kolumny, columns=kolumny, dtype=float)
    for k1 in kolumny:
        for k2 in kolumny:
            _, p = stats.pearsonr(df[k1], df[k2])
            wynik.loc[k1, k2] = p
    return wynik


macierz_p(klienci, kolumny_liczbowe)
```
</details>

> 🔥 **Wyzwanie: fałszywa korelacja przez zmienną ukrytą**
>
> Zbuduj dwie SZTUCZNE zmienne, obie zależne od tej samej trzeciej, ukrytej zmiennej `Z` (np. `Z = rng.normal(0, 1, size=500)`, `X = Z + szum1`, `Y = Z + szum2`, gdzie `szum1`/`szum2` to niezależny szum losowy). Policz korelację między `X` i `Y` -- powinna wyjść wyraźnie różna od zera, mimo że `X` w ogóle nie wpływa przyczynowo na `Y` (oba są tylko efektami `Z`). To namacalny przykład punktu 3 z sekcji 5.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
rng2 = np.random.default_rng(9)
z_ukryte = rng2.normal(0, 1, size=500)
x_falszywe = z_ukryte + rng2.normal(0, 0.5, size=500)
y_falszywe = z_ukryte + rng2.normal(0, 0.5, size=500)

r_falszywe, p_falszywe = stats.pearsonr(x_falszywe, y_falszywe)
print(f"r(X, Y) = {r_falszywe:.3f}, p = {p_falszywe:.2e}")
print("X nie wplywa przyczynowo na Y -- cala korelacja pochodzi ze wspolnej zaleznosci od Z")
```
</details>

## Co dalej?

Korelacja mówi, że związek istnieje i jak silny jest. **Regresja liniowa** w
**Module 12** pójdzie o krok dalej: zbuduje model matematyczny pozwalający
PRZEWIDZIEĆ wartość jednej zmiennej na podstawie drugiej (albo kilku innych), z
konkretnym równaniem i miarą tego, jak dobrze model pasuje do danych.